In [37]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.chdir('/home/jovyan/kuratov/data/test_time_gd/')

import torch

In [42]:
from kv_dataset_utils import generate_sequence, create_tokenizer

tokenizer = create_tokenizer()

num_kv_pairs = 1
k_length = 2
v_length = 2
n_segments = 2
min_segment_len = 4
max_segment_len = 16




sample = generate_sequence(num_kv_pairs=num_kv_pairs, k_length=k_length, v_length=v_length, n_segments=n_segments,
                           min_segment_len=min_segment_len, max_segment_len=max_segment_len)
print(sample['sequence'] + sample['target'])

print('decode(tokenizer.encode):')
print(tokenizer.decode(tokenizer(sample['sequence'] + sample['target'],
                                 padding=True, pad_to_multiple_of=8).input_ids).replace(' ', ''))

7!IU:pk!OSJ|u1XXdo0cZuzren|?!IU:pk!|
decode(tokenizer.encode):
7!IU:pk!OSJ|u1XXdo0cZuzren|?!IU:pk!|[PAD][PAD][PAD][PAD]


In [6]:
from transformers import GPT2Config

config = GPT2Config.from_pretrained('gpt2')
config.n_layer = 4
config.n_head = 4
config.n_embd = 128
config.vocab_size = 128
config.pad_token_id = vocab['[PAD]']
config.bos_token_id = vocab['[BOS]']
config.eos_token_id = vocab['[EOS]']

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class KVDataset(Dataset):
    def __init__(self, num_samples, **gen_params):
        self.samples = [
            generate_sequence(**gen_params) 
            for _ in range(num_samples)
        ]
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        input_seq = sample['sequence']
        target_seq = sample['target']
        
        return {
            'input_seq': input_seq,
            'target_seq': target_seq,
        }

In [8]:
train_dataset = KVDataset(1000, 
                         num_kv_pairs=num_kv_pairs, 
                         k_length=k_length, 
                         v_length=v_length, 
                         n_segments=n_segments,
                         min_segment_len=min_segment_len, 
                         max_segment_len=max_segment_len)

def collate_fn(batch):
    input_seq = [item['input_seq'] for item in batch]
    target_seq = [item['target_seq'] for item in batch]
    seq = [item['input_seq'] + item['target_seq'] for item in batch]
    input_ids = tokenizer(seq, return_tensors="pt", add_special_tokens=False,
                          padding=True, pad_to_multiple_of=8).input_ids
    # add labels_mask
    # input_seq: 0, target_seq: 1, seq = input_seq + target_seq
    labels_mask = torch.zeros_like(input_ids)
    for i, item in enumerate(batch):
        input_seq_len = len(item['input_seq'])
        target_seq_len = len(item['target_seq'])
        # +1 as bos token was added to the beginning, +2 as eos is in the end and we want to predict it
        labels_mask[i, input_seq_len:input_seq_len+target_seq_len] = 1

    labels = input_ids * labels_mask + (1 - labels_mask) * -100
    return {
        'input_seq': input_seq,
        'target_seq': target_seq,
        'input_ids': input_ids,
        'labels': labels,
        'labels_mask': labels_mask,
    }

batch_size = 2
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

In [9]:
for batch in train_dataloader:
    for k in batch:
        print(k, batch[k])
    break
    


input_seq ['!Tk:K2!hL|nRX21Ii|?!Tk:', 'MFkAnXE|pgLMgQ!v3:k4!V|?!v3:']
target_seq ['K2!|', 'k4!|']
input_ids tensor([[66, 49, 14, 68, 40, 58, 66, 11, 41, 69, 17, 47, 53, 58, 57, 38, 12, 69,
         67, 66, 49, 14, 68, 40, 58, 66, 69,  0,  0,  0,  0,  0],
        [42, 35, 14, 30, 17, 53, 34, 69, 19, 10, 41, 42, 10, 46, 66, 25, 59, 68,
         14, 60, 66, 51, 69, 67, 66, 25, 59, 68, 14, 60, 66, 69]])
labels tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,   40,
           58,   66,   69, -100, -100, -100, -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100,   14,   60,   66,   69]])
labels_mask tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
         1, 1, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0

In [ ]:
# # Training with HuggingFace Trainer
from typing import Any, Dict, Optional, Union
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset
import numpy as np
from tqdm import tqdm


from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel(config)

num_kv_pairs = 1
k_length = 2
v_length = 2
n_segments = 4
min_segment_len = 2
max_segment_len = 16

# Create dataset and dataloader
train_dataset = KVDataset(100000,
                         num_kv_pairs=num_kv_pairs,
                         k_length=k_length,
                         v_length=v_length,
                         n_segments=n_segments,
                         min_segment_len=min_segment_len,
                         max_segment_len=max_segment_len)

valid_dataset = KVDataset(1000,
                         num_kv_pairs=num_kv_pairs,
                         k_length=k_length,
                         v_length=v_length,
                         n_segments=n_segments,
                         min_segment_len=min_segment_len,
                         max_segment_len=max_segment_len)

def collate_fn(batch):
    input_seq = [item['input_seq'] for item in batch]
    target_seq = [item['target_seq'] for item in batch]
    seq = [item['input_seq'] + item['target_seq'] for item in batch]
    input_ids = tokenizer(seq, return_tensors="pt", add_special_tokens=False,
                          padding=True, pad_to_multiple_of=8).input_ids
    # add labels_mask
    # input_seq: 0, target_seq: 1, seq = input_seq + target_seq
    labels_mask = torch.zeros_like(input_ids)
    for i, item in enumerate(batch):
        input_seq_len = len(item['input_seq'])
        target_seq_len = len(item['target_seq'])
        labels_mask[i, input_seq_len:input_seq_len+target_seq_len] = 1

    labels = input_ids * labels_mask + (1 - labels_mask) * -100

    return {
        # 'input_seq': input_seq,
        # 'target_seq': target_seq,
        'input_ids': input_ids,
        'labels': labels,
        # 'labels_mask': labels_mask,
    }

# target sequence looks like: "XXXX!|"
# let's not count ! and | in the accuracy calculation
ignore_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in ['!', '|']]

# Define compute_metrics function for token-level accuracy and loss
def compute_metrics(eval_pred):
    logits, labels, inputs = eval_pred.predictions, eval_pred.label_ids, eval_pred.inputs

    logits = logits[..., :-1, :]
    labels = labels[..., 1:]
    predictions = np.argmax(logits, axis=-1)
    
    # Create a mask for tokens that are not padding (-100) and ignored tokens (like ! and |)
    mask = (labels != -100)
    for t_id in ignore_token_ids:
        mask &= (labels != t_id)
    
    # Calculate token-level accuracy only on content tokens
    masked_predictions = predictions[mask]
    masked_labels = labels[mask]
    
    accuracy = (masked_predictions == masked_labels).mean()
    
    # get exact_match per-sample accuracy, ignore masked tokens
    # predictions.shape = (batch_size, seq_len)
    exact_match = np.mean([
        np.all(pred[mask[i]] == lab[mask[i]])
        for i, (pred, lab) in enumerate(zip(predictions, labels))
        if np.any(mask[i])  # Skip samples that are all masked
    ])

    for pred, label, inp in zip(predictions[:5], labels[:5], inputs[:5]):
        mask = (label != -100)
        pred = pred[mask]
        inp[inp==-100] = 0
        label[label==-100] = 0
        print('i:', tokenizer.decode(inp, skip_special_tokens=True).replace(' ', ''))
        print('p:', tokenizer.decode(pred, skip_special_tokens=True).replace(' ', ''))
        print('t:', tokenizer.decode(label, skip_special_tokens=True).replace(' ', ''))
        print('-' * 50)

    return {
        "token_accuracy": float(accuracy),
        "exact_match": float(exact_match),
    }


class CustomTrainer(Trainer):

    def create_scheduler(self, num_training_steps: int, optimizer: torch.optim.Optimizer = None):
        num_training_steps = int(num_training_steps / 0.9)  # to make final lr not zero, for linear it is lr/10.
        return super().create_scheduler(num_training_steps, optimizer)

    def log(self, logs: dict[str, float], start_time: Optional[float] = None) -> None:
        # log early stopping patience
        for cb in self.callback_handler.callbacks:
            if isinstance(cb, EarlyStoppingCallback):
                logs['patience'] = cb.early_stopping_patience_counter
                break
        return super().log(logs, start_time=start_time)


# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    weight_decay=0.001,
    learning_rate=1e-04,
    lr_scheduler_type='constant_with_warmup',
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="token_accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    include_num_input_tokens_seen=True,
    include_for_metrics=['inputs']
)

# Initialize Trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=15),],
)

# Train the model
trainer.train()

In [34]:
len(tokenizer.vocab)

70

In [35]:
len(ALPHABET)

62

In [36]:
string.printable

'0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~ \t\n\r\x0b\x0c'